# Baseline on the Synthetic Dataset
### This Jupyter Notebook simulates 7 baseline methods on the synthetic data.

## Import libraries

In [1]:
import numpy as np
import torch
import torch.nn as nn
import torch.optim as optim

from torch.utils.data import DataLoader
from sklearn.metrics import accuracy_score
from sklearn.preprocessing import MinMaxScaler, StandardScaler, RobustScaler

from skmultiflow.trees import HoeffdingAdaptiveTreeClassifier
from skmultiflow.meta import AdaptiveRandomForestClassifier

from model import NormalNN, EarlyStopping, NNClassifier

from Sequential import FSCDS_SFS, DSFSC_SFS

from utils import prepare_data

## Load data

In [2]:
x_all = np.load('./dataset/SEA_data.npy')
y_all = np.load('./dataset/SEA_label.npy')
concept_drifts = np.load('./dataset/SEA_drift_point.npy')

In [3]:
cuda = True if torch.cuda.is_available() else False

if cuda:
    torch.cuda.set_device(0)

## Baseline(All Data)

In [4]:
lr = 0.001
num = int(concept_drifts[-1]/len(concept_drifts))

all_test_acc = []
all_test_std = []

all_dataset = []
all_feature = []
all_usage = []

# Incremental training and evaluation from the 7th data segment(n=6)
for n in range(6, len(concept_drifts)):
    n_dataset = n+1
    n_feature = x_all.shape[1]
    
    print("test segment: ", n_dataset)
    
    # Set the train, valid, test data
    scaler = MinMaxScaler()
    x_prev = scaler.fit_transform(x_all[:concept_drifts[n-1]+int(num*0.1)])
    x_target = scaler.transform(x_all[concept_drifts[n-1]+int(num*0.1):concept_drifts[n]])
    x_all_temp = np.concatenate((x_prev, x_target))

    dataset_final = range(n+1)
    feature_final = range(n_feature)

    train_ds, valid_ds, test_ds = prepare_data(n, x_all_temp, y_all, concept_drifts, dataset_final, feature_final)
    train_loader = DataLoader(train_ds, batch_size=128, shuffle=True)
    valid_loader = DataLoader(valid_ds, batch_size=128, shuffle=True)
    test_loader = DataLoader(test_ds, batch_size=128, shuffle=True)
    
    test_score_li = []
    num_dataset_li = []
    num_feature_li = []
    usage_li = []
    
    # Repeat experiment with 5 different seeds
    for s in range(5):
        
        # ---------------------
        #  Initialize model, optimizer, and criterion
        # ---------------------
        model = NormalNN(input_features=n_feature, seed=s)
        model = model.cuda()
        optimizer_config = {"lr": lr}
        clf = NNClassifier(model, nn.BCELoss(reduction='mean'), optim.Adam, optimizer_config)
        
        # ---------------------
        #  Model training
        # ---------------------
        clf.fit({"train": train_loader, "val": valid_loader}, epochs=2000, earlystop_path='checkpoint_SEA_All.pt')

        num_dataset = len(dataset_final)
        num_feature = len(feature_final)
        usage = float(num_dataset*num_feature/(n_dataset*n_feature))*100
        
        num_dataset_li.append(num_dataset)
        num_feature_li.append(num_feature)
        usage_li.append(usage)

        test_output, test_loss = clf.evaluate(test_loader)
        test_score = accuracy_score(test_output['true_y'], test_output['output'])
        test_score_li.append(test_score) 
        
    print('test acc: avg %f, std %f' %(np.mean(test_score_li), np.std(test_score_li)))
    all_test_acc.append(np.mean(test_score_li))
    all_test_std.append(np.std(test_score_li))
    
    print('data usage(#feature, #segment, %%total): (%f, %f, %f%%)' 
          %(np.mean(num_feature_li), np.mean(num_dataset_li), np.mean(usage_li)))
    all_dataset.append(np.mean(num_dataset_li))
    all_feature.append(np.mean(num_feature_li))
    all_usage.append(np.mean(usage_li))
    
    print("----------------------------------------------------------------------------")
    
print('overall test acc: avg %f, std %f' %(np.mean(all_test_acc), np.mean(all_test_std)))
print('overall data usage(#feature, #segment, %%total): (%f, %f, %f%%)' 
      %(np.mean(all_feature), np.mean(all_dataset), np.mean(all_usage)))

test segment:  7
test acc: avg 0.462222, std 0.039068
data usage(#feature, #segment, %total): (3.000000, 7.000000, 100.000000%)
----------------------------------------------------------------------------
test segment:  8
test acc: avg 0.634889, std 0.010119
data usage(#feature, #segment, %total): (3.000000, 8.000000, 100.000000%)
----------------------------------------------------------------------------
test segment:  9
test acc: avg 0.868111, std 0.009300
data usage(#feature, #segment, %total): (3.000000, 9.000000, 100.000000%)
----------------------------------------------------------------------------
overall test acc: avg 0.655074, std 0.019496
overall data usage(#feature, #segment, %total): (3.000000, 8.000000, 100.000000%)


## Baseline(Current Seg.)

In [5]:
lr = 0.001
num = int(concept_drifts[-1]/len(concept_drifts))

all_test_acc = []
all_test_std = []

all_dataset = []
all_feature = []
all_usage = []

# Incremental training and evaluation from the 7th data segment(n=6) 
for n in range(6, len(concept_drifts)):
    n_dataset = n+1
    n_feature = x_all.shape[1]

    print("test segment: ", n_dataset)

    # Set the train, valid, test data
    scaler = MinMaxScaler()
    x_useless = x_all[:concept_drifts[n-1]]
    x_prev = scaler.fit_transform(x_all[concept_drifts[n-1]:concept_drifts[n-1]+int(num*0.1)])
    x_target = scaler.transform(x_all[concept_drifts[n-1]+int(num*0.1):concept_drifts[n]])
    x_all_temp = np.concatenate((x_useless, x_prev, x_target))

    dataset_final = [n]
    feature_final = range(n_feature)

    train_ds, valid_ds, test_ds = prepare_data(n, x_all_temp, y_all, concept_drifts, dataset_final, feature_final)

    train_loader = DataLoader(train_ds, batch_size=128, shuffle=True)
    valid_loader = DataLoader(valid_ds, batch_size=128, shuffle=True)
    test_loader = DataLoader(test_ds, batch_size=128, shuffle=True)

    test_score_li = []
    num_dataset_li = []
    num_feature_li = []
    usage_li = []
    
    # Repeat experiment with 5 different seeds
    for s in range(5):
        
        # ---------------------
        #  Initialize model, optimizer, and criterion
        # ---------------------
        model = NormalNN(input_features=n_feature, seed=s)
        model = model.cuda()
        optimizer_config = {"lr": lr}
        clf = NNClassifier(model, nn.BCELoss(reduction='mean'), optim.Adam, optimizer_config)
        
        # ---------------------
        #  Model training
        # ---------------------
        clf.fit({"train": train_loader, "val": valid_loader}, epochs=2000, earlystop_path='checkpoint_SEA_Current.pt')

        num_dataset = len(dataset_final)
        num_feature = len(feature_final)
        usage = float(num_dataset*num_feature/(n_dataset*n_feature))*100

        num_dataset_li.append(num_dataset)
        num_feature_li.append(num_feature)
        usage_li.append(usage)
        
        test_output, test_loss = clf.evaluate(test_loader)
        test_score = accuracy_score(test_output['true_y'], test_output['output'])
        test_score_li.append(test_score)
        
    print('test acc: avg %f, std %f' %(np.mean(test_score_li), np.std(test_score_li)))
    all_test_acc.append(np.mean(test_score_li))
    all_test_std.append(np.std(test_score_li))
    
    print('data usage(#feature, #segment, %%total): (%f, %f, %f%%)' 
          %(np.mean(num_feature_li), np.mean(num_dataset_li), np.mean(usage_li)))
    all_dataset.append(np.mean(num_dataset_li))
    all_feature.append(np.mean(num_feature_li))
    all_usage.append(np.mean(usage_li))
    
    print("----------------------------------------------------------------------------")  

print('overall test acc: avg %f, std %f' %(np.mean(all_test_acc), np.mean(all_test_std)))
print('overall data usage(#feature, #segment, %%total): (%f, %f, %f%%)' 
      %(np.mean(all_feature), np.mean(all_dataset), np.mean(all_usage)))

test segment:  7
test acc: avg 0.819222, std 0.002643
data usage(#feature, #segment, %total): (3.000000, 1.000000, 14.285714%)
----------------------------------------------------------------------------
test segment:  8
test acc: avg 0.723222, std 0.084700
data usage(#feature, #segment, %total): (3.000000, 1.000000, 12.500000%)
----------------------------------------------------------------------------
test segment:  9
test acc: avg 0.885222, std 0.002449
data usage(#feature, #segment, %total): (3.000000, 1.000000, 11.111111%)
----------------------------------------------------------------------------
overall test acc: avg 0.809222, std 0.029931
overall data usage(#feature, #segment, %total): (3.000000, 1.000000, 12.632275%)


## Baseline(Retrain)

In [6]:
num = int(concept_drifts[-1]/len(concept_drifts))

all_test_acc = []
all_test_std = []

all_dataset = []
all_feature = []
all_usage = []

# Incremental training and evaluation from the 7th data segment(n=6) 
for n in range(6, len(concept_drifts)):   
    n_dataset = n+1
    n_feature = x_all.shape[1]
    
    print("test segment: ", n_dataset)

    # Set the train, valid, test data
    scaler = MinMaxScaler()
    x_useless = x_all[:concept_drifts[n-1]]
    x_prev = scaler.fit_transform(x_all[concept_drifts[n-1]:concept_drifts[n-1]+int(num*0.1)])
    x_target = scaler.transform(x_all[concept_drifts[n-1]+int(num*0.1):concept_drifts[n]])
    x_all_temp = np.concatenate((x_useless, x_prev, x_target))

    dataset_final = [n]
    feature_final = range(n_feature)
    
    x_temp = x_all_temp[concept_drifts[n-1]:concept_drifts[n]]
    y_temp = y_all[concept_drifts[n-1]:concept_drifts[n]]

    indices = list(range(len(x_temp)))
    split1 = int(len(x_temp)*0.01)
    split2 = int(len(x_temp)*0.1)
    train_indices, valid_indices, test_indices = indices[:split1], indices[split1:split2], indices[split2:]

    x_train = np.concatenate((x_temp[train_indices], x_temp[valid_indices]))
    y_train = np.concatenate((y_temp[train_indices], y_temp[valid_indices])).reshape(-1,1)
    
    x_test = x_temp[test_indices]
    y_test = y_temp[test_indices].reshape(-1,1)
    
    test_score_li = []
    num_dataset_li = []
    num_feature_li = []
    usage_li = []
    
    # Repeat experiment with 5 different seeds
    for s in range(5):
        # ---------------------
        #  Initialize model, optimizer, and criterion
        # ---------------------
        arf = HoeffdingAdaptiveTreeClassifier(random_state=s)

        # ---------------------
        #  Model training
        # ---------------------
        for i in range(len(x_train)):
            X = x_train[i]
            y = y_train[i]
            arf.partial_fit([X], y)
            
        num_dataset = len(dataset_final)
        num_feature = len(feature_final)
        usage = float(num_dataset*num_feature/(n_dataset*n_feature))*100

        num_dataset_li.append(num_dataset)
        num_feature_li.append(num_feature)
        usage_li.append(usage)

        n_samples = 0
        correct_cnt = 0

        for i in range(len(x_test)):
            X = x_test[i]
            y = y_test[i]
            y_pred = arf.predict([X])
            if y[0] == y_pred[0]:
                correct_cnt += 1
            n_samples += 1
        
        test_score_li.append(float(correct_cnt/n_samples))

    print('test acc: avg %f, std %f' %(np.mean(test_score_li), np.std(test_score_li)))
    all_test_acc.append(np.mean(test_score_li))
    all_test_std.append(np.std(test_score_li))
    
    print('data usage(#feature, #segment, %%total): (%f, %f, %f%%)' 
          %(np.mean(num_feature_li), np.mean(num_dataset_li), np.mean(usage_li)))
    all_dataset.append(np.mean(num_dataset_li))
    all_feature.append(np.mean(num_feature_li))
    all_usage.append(np.mean(usage_li))
    
    print("----------------------------------------------------------------------------")
    
print('overall test acc: avg %f, std %f' %(np.mean(all_test_acc), np.mean(all_test_std)))
print('overall data usage(#feature, #segment, %%total): (%f, %f, %f%%)' 
      %(np.mean(all_feature), np.mean(all_dataset), np.mean(all_usage)))

test segment:  7
test acc: avg 0.882333, std 0.005145
data usage(#feature, #segment, %total): (3.000000, 1.000000, 14.285714%)
----------------------------------------------------------------------------
test segment:  8
test acc: avg 0.833889, std 0.010813
data usage(#feature, #segment, %total): (3.000000, 1.000000, 12.500000%)
----------------------------------------------------------------------------
test segment:  9
test acc: avg 0.879556, std 0.009470
data usage(#feature, #segment, %total): (3.000000, 1.000000, 11.111111%)
----------------------------------------------------------------------------
overall test acc: avg 0.865259, std 0.008476
overall data usage(#feature, #segment, %total): (3.000000, 1.000000, 12.632275%)


## Baseline(Adjust)

In [7]:
num = int(concept_drifts[-1]/len(concept_drifts))

all_test_acc = []
all_test_std = []

all_dataset = []
all_feature = []
all_usage = []

# Incremental training and evaluation from the 7th data segment(n=6) 
for n in range(6, len(concept_drifts)):
    n_dataset = n+1
    n_feature = x_all.shape[1]
    
    print("n: ", n)
    
    # Set the train, valid, test data
    scaler = MinMaxScaler()
    x_prev = scaler.fit_transform(x_all[:concept_drifts[n-1]+int(num*0.1)])
    x_target = scaler.transform(x_all[concept_drifts[n-1]+int(num*0.1):concept_drifts[n]])
    x_all_temp = np.concatenate((x_prev, x_target))

    dataset_final = range(n+1)
    feature_final = range(n_feature)
    
    x_temp = x_all_temp[concept_drifts[n-1]:concept_drifts[n]]
    y_temp = y_all[concept_drifts[n-1]:concept_drifts[n]]

    indices = list(range(len(x_temp)))
    split1 = int(len(x_temp)*0.01)
    split2 = int(len(x_temp)*0.1)
    train_indices, valid_indices, test_indices = indices[:split1], indices[split1:split2], indices[split2:]

    x_train = np.concatenate((x_all_temp[:concept_drifts[n-1]], x_temp[train_indices], x_temp[valid_indices]))
    y_train = np.concatenate((y_all[:concept_drifts[n-1]], y_temp[train_indices], y_temp[valid_indices])).reshape(-1,1)
    
    x_test = x_temp[test_indices]
    y_test = y_temp[test_indices].reshape(-1,1)
    
    test_score_li = []
    num_dataset_li = []
    num_feature_li = []
    usage_li = []
    
    # Repeat experiment with 5 different seeds
    for s in range(5):
        # ---------------------
        #  Initialize model, optimizer, and criterion
        # ---------------------
        arf = HoeffdingAdaptiveTreeClassifier(random_state=s)

        # ---------------------
        #  Model training
        # ---------------------
        for i in range(len(x_train)):
            X = x_train[i]
            y = y_train[i]
            arf.partial_fit([X], y)
            
        num_dataset = len(dataset_final)
        num_feature = len(feature_final)
        usage = float(num_dataset*num_feature/(n_dataset*n_feature))*100

        num_dataset_li.append(num_dataset)
        num_feature_li.append(num_feature)
        usage_li.append(usage)

        n_samples = 0
        correct_cnt = 0

        for i in range(len(x_test)):
            X = x_test[i]
            y = y_test[i]
            y_pred = arf.predict([X])
            if y[0] == y_pred[0]:
                correct_cnt += 1
            n_samples += 1
        
        test_score_li.append(float(correct_cnt/n_samples))

    print('test acc: avg %f, std %f' %(np.mean(test_score_li), np.std(test_score_li)))
    all_test_acc.append(np.mean(test_score_li))
    all_test_std.append(np.std(test_score_li))
    
    print('data usage(#feature, #segment, %%total): (%f, %f, %f%%)' 
          %(np.mean(num_feature_li), np.mean(num_dataset_li), np.mean(usage_li)))
    all_dataset.append(np.mean(num_dataset_li))
    all_feature.append(np.mean(num_feature_li))
    all_usage.append(np.mean(usage_li))
    
    print("----------------------------------------------------------------------------")
    
print('overall test acc: avg %f, std %f' %(np.mean(all_test_acc), np.mean(all_test_std)))
print('overall data usage(#feature, #segment, %%total): (%f, %f, %f%%)' 
      %(np.mean(all_feature), np.mean(all_dataset), np.mean(all_usage)))

n:  6
test acc: avg 0.350000, std 0.000000
data usage(#feature, #segment, %total): (3.000000, 7.000000, 100.000000%)
----------------------------------------------------------------------------
n:  7
test acc: avg 0.644667, std 0.017542
data usage(#feature, #segment, %total): (3.000000, 8.000000, 100.000000%)
----------------------------------------------------------------------------
n:  8
test acc: avg 0.608111, std 0.030331
data usage(#feature, #segment, %total): (3.000000, 9.000000, 100.000000%)
----------------------------------------------------------------------------
overall test acc: avg 0.534259, std 0.015958
overall data usage(#feature, #segment, %total): (3.000000, 8.000000, 100.000000%)


## Baseline(Ensemble)

In [8]:
num = int(concept_drifts[-1]/len(concept_drifts))

all_test_acc = []
all_test_std = []

all_dataset = []
all_feature = []
all_usage = []

# Incremental training and evaluation from the 7th data segment(n=6) 
for n in range(6, len(concept_drifts)):
    n_dataset = n+1
    n_feature = x_all.shape[1]
    
    print("n: ", n)
    
    # Set the train, valid, test data
    scaler = MinMaxScaler()
    x_prev = scaler.fit_transform(x_all[:concept_drifts[n-1]+int(num*0.1)])
    x_target = scaler.transform(x_all[concept_drifts[n-1]+int(num*0.1):concept_drifts[n]])
    x_all_temp = np.concatenate((x_prev, x_target))

    dataset_final = range(n+1)
    feature_final = range(n_feature)
    
    x_temp = x_all_temp[concept_drifts[n-1]:concept_drifts[n]]
    y_temp = y_all[concept_drifts[n-1]:concept_drifts[n]]

    indices = list(range(len(x_temp)))
    split1 = int(len(x_temp)*0.01)
    split2 = int(len(x_temp)*0.1)
    train_indices, valid_indices, test_indices = indices[:split1], indices[split1:split2], indices[split2:]

    x_train = np.concatenate((x_all_temp[:concept_drifts[n-1]], x_temp[train_indices], x_temp[valid_indices]))
    y_train = np.concatenate((y_all[:concept_drifts[n-1]], y_temp[train_indices], y_temp[valid_indices])).reshape(-1,1)
    
    x_test = x_temp[test_indices]
    y_test = y_temp[test_indices].reshape(-1,1)
    
    test_score_li = []
    num_dataset_li = []
    num_feature_li = []
    usage_li = []
    
    # Repeat experiment with 5 different seeds
    for s in range(5):
        # ---------------------
        #  Initialize model, optimizer, and criterion
        # ---------------------
        arf = AdaptiveRandomForestClassifier(random_state=s)

        # ---------------------
        #  Model training
        # ---------------------
        for i in range(len(x_train)):
            X = x_train[i]
            y = y_train[i]
            arf.partial_fit([X], y)
            
        num_dataset = len(dataset_final)
        num_feature = len(feature_final)
        usage = float(num_dataset*num_feature/(n_dataset*n_feature))*100

        num_dataset_li.append(num_dataset)
        num_feature_li.append(num_feature)
        usage_li.append(usage)

        n_samples = 0
        correct_cnt = 0

        for i in range(len(x_test)):
            X = x_test[i]
            y = y_test[i]
            y_pred = arf.predict([X])
            if y[0] == y_pred[0]:
                correct_cnt += 1
            n_samples += 1
        
        test_score_li.append(float(correct_cnt/n_samples))

    print('test acc: avg %f, std %f' %(np.mean(test_score_li), np.std(test_score_li)))
    all_test_acc.append(np.mean(test_score_li))
    all_test_std.append(np.std(test_score_li))
    
    print('data usage(#feature, #segment, %%total): (%f, %f, %f%%)' 
          %(np.mean(num_feature_li), np.mean(num_dataset_li), np.mean(usage_li)))
    all_dataset.append(np.mean(num_dataset_li))
    all_feature.append(np.mean(num_feature_li))
    all_usage.append(np.mean(usage_li))
    
    print("----------------------------------------------------------------------------")
    
print('overall test acc: avg %f, std %f' %(np.mean(all_test_acc), np.mean(all_test_std)))
print('overall data usage(#feature, #segment, %%total): (%f, %f, %f%%)' 
      %(np.mean(all_feature), np.mean(all_dataset), np.mean(all_usage)))

n:  6
test acc: avg 0.840667, std 0.016638
data usage(#feature, #segment, %total): (3.000000, 7.000000, 100.000000%)
----------------------------------------------------------------------------
n:  7
test acc: avg 0.695000, std 0.011664
data usage(#feature, #segment, %total): (3.000000, 8.000000, 100.000000%)
----------------------------------------------------------------------------
n:  8
test acc: avg 0.737667, std 0.049468
data usage(#feature, #segment, %total): (3.000000, 9.000000, 100.000000%)
----------------------------------------------------------------------------
overall test acc: avg 0.757778, std 0.025924
overall data usage(#feature, #segment, %total): (3.000000, 8.000000, 100.000000%)


## Baseline(FSC->DS)

In [9]:
lr = 0.001
num = int(concept_drifts[-1]/len(concept_drifts))

all_test_acc = []
all_test_std = []

all_dataset = []
all_feature = []
all_usage = []

# Incremental training and evaluation from the 7th data segment(n=6) 
for n in range(6, len(concept_drifts)):
    
    n_dataset = n+1
    n_feature = x_all.shape[1]

    # ---------------------
    #  Define FSC->DS (sequential two-step approach)
    # ---------------------
    sequential = FSCDS_SFS(x_all, y_all, n_dataset, n_feature, concept_drifts, num)
    
    # ---------------------
    #  Data calibration and sampling
    # ---------------------
    scaler_list = [MinMaxScaler(), StandardScaler(), RobustScaler()]
    x_all_cali, y_all_cali = sequential.calibration(n, scaler_list)
    x_sample, y_sample, concept_drifts_sample = sequential.sampling(n, x_all_cali, y_all_cali)
    
    print("test segment: ", n_dataset)

    test_score_li = []
    num_dataset_li = []
    num_feature_li = []
    usage_li = []
    
    # Repeat experiment with 5 different seeds
    for s in range(5):

        # ---------------------
        #  Feature Selection and Calibration(FSC)
        # ---------------------
        chromo_df_pcos,score_pcos = sequential.generation_fs(n, x_sample, y_sample, concept_drifts_sample, 
                                                             lr, n_gen=50, seed=s)
        
        # ---------------------
        #  Data Selection(DS)
        # ---------------------
        chromo_df_pcos,score_pcos = sequential.generation_ds(n, x_sample, y_sample, concept_drifts_sample, 
                                                             chromo_df_pcos, lr, n_gen=50, seed=s)
        
        # ---------------------
        #  Selected and calibrated (data segment, feature) result
        # ---------------------
        dataset_final = []
        feature_final = []
        
        for i in range(n_dataset):
            if chromo_df_pcos[i]:
                dataset_final.append(i)
                
        for j in range(n_dataset, n_dataset+n_feature*(1+len(scaler_list))):
            if chromo_df_pcos[j]:
                feature_final.append(j-n_dataset)
                
        num_dataset = len(dataset_final)
        num_feature = len(feature_final)
        usage = float(num_dataset*num_feature/(n_dataset*n_feature))*100
        
        num_dataset_li.append(num_dataset)
        num_feature_li.append(num_feature)
        usage_li.append(usage)
        
        # ---------------------
        #  Construct new train data with Quilt result 
        # ---------------------
        train_ds, valid_ds, test_ds = prepare_data(n, x_all_cali, y_all_cali, concept_drifts, 
                                                   dataset_final, feature_final)
        train_loader = DataLoader(train_ds, batch_size=128, shuffle=True)
        valid_loader = DataLoader(valid_ds, batch_size=128, shuffle=True)
        test_loader = DataLoader(test_ds, batch_size=128, shuffle=True)

        # ---------------------
        #  Initialize model, optimizer, and criterion
        # ---------------------
        model = NormalNN(input_features=len(feature_final), seed=s)
        model = model.cuda()
        optimizer_config = {"lr": lr}
        clf = NNClassifier(model, nn.BCELoss(reduction='mean'), optim.Adam, optimizer_config)
        
        # ---------------------
        #  Model training
        # ---------------------
        clf.fit({"train": train_loader, "val": valid_loader}, epochs=2000, earlystop_path='checkpoint_SEA_FSCDS.pt')
        
        test_output, test_loss = clf.evaluate(test_loader)
        test_score = accuracy_score(test_output['true_y'], test_output['output'])
        test_score_li.append(test_score)
        
    print('test acc: avg %f, std %f' %(np.mean(test_score_li), np.std(test_score_li)))
    all_test_acc.append(np.mean(test_score_li))
    all_test_std.append(np.std(test_score_li))
    
    print('data usage(#feature, #segment, %%total): (%f, %f, %f%%)' 
          %(np.mean(num_feature_li), np.mean(num_dataset_li), np.mean(usage_li)))
    all_dataset.append(np.mean(num_dataset_li))
    all_feature.append(np.mean(num_feature_li))
    all_usage.append(np.mean(usage_li))
    
    print("----------------------------------------------------------------------------")
    
print('overall test acc: avg %f, std %f' %(np.mean(all_test_acc), np.mean(all_test_std)))
print('overall data usage(#feature, #segment, %%total): (%f, %f, %f%%)' 
      %(np.mean(all_feature), np.mean(all_dataset), np.mean(all_usage)))

test segment:  7
test acc: avg 0.847889, std 0.068612
data usage(#feature, #segment, %total): (1.800000, 2.200000, 19.047619%)
----------------------------------------------------------------------------
test segment:  8
test acc: avg 0.866889, std 0.005979
data usage(#feature, #segment, %total): (2.600000, 1.200000, 12.500000%)
----------------------------------------------------------------------------
test segment:  9
test acc: avg 0.904667, std 0.008116
data usage(#feature, #segment, %total): (2.400000, 3.600000, 32.592593%)
----------------------------------------------------------------------------
overall test acc: avg 0.873148, std 0.027569
overall data usage(#feature, #segment, %total): (2.266667, 2.333333, 21.380071%)


## Baseline(DS->FSC)

In [10]:
lr = 0.001
num = int(concept_drifts[-1]/len(concept_drifts))

all_test_acc = []
all_test_std = []

all_dataset = []
all_feature = []
all_usage = []

# Incremental training and evaluation from the 7th data segment(n=6) 
for n in range(6, len(concept_drifts)):
    
    n_dataset = n+1
    n_feature = x_all.shape[1]

    # ---------------------
    #  Define DS->FSC (sequential two-step approach)
    # ---------------------
    sequential = DSFSC_SFS(x_all, y_all, n_dataset, n_feature, concept_drifts, num)
    
    scaler = MinMaxScaler()
    x_all_scale, y_all_scale = sequential.scaling(n, scaler)
    x_sample, y_sample, concept_drifts_sample = sequential.sampling(n, x_all_scale, y_all_scale)
    
    print("test segment: ", n_dataset)

    test_score_li = []
    num_dataset_li = []
    num_feature_li = []
    usage_li = []
    
    # Repeat experiment with 5 different seeds
    for s in range(5):

        # ---------------------
        #  Data Selection(DS)
        # ---------------------
        chromo_df_pcos,score_pcos = sequential.generation_ds(n, x_sample, y_sample, concept_drifts_sample, 
                                                             lr, n_gen=50, seed=s)
        
        # ---------------------
        #  Data calibration and sampling
        # ---------------------
        scaler_list = [MinMaxScaler(), StandardScaler(), RobustScaler()]
        x_all_cali, y_all_cali = sequential.calibration(n, scaler_list)
        x_sample, y_sample, concept_drifts_sample = sequential.sampling(n, x_all_cali, y_all_cali)
        
        # ---------------------
        #  Feature Selection and Calibration(FSC)
        # ---------------------
        chromo_df_pcos,score_pcos = sequential.generation_fs(n, x_sample, y_sample, concept_drifts_sample, 
                                                             chromo_df_pcos, lr, n_gen=50, seed=s)
        
        # ---------------------
        #  Selected and calibrated (data segment, feature) result
        # ---------------------
        dataset_final = []
        feature_final = []
        
        for i in range(n_dataset):
            if chromo_df_pcos[i]:
                dataset_final.append(i)
                
        for j in range(n_dataset, n_dataset+n_feature*(1+len(scaler_list))):
            if chromo_df_pcos[j]:
                feature_final.append(j-n_dataset)
                
        num_dataset = len(dataset_final)
        num_feature = len(feature_final)
        usage = float(num_dataset*num_feature/(n_dataset*n_feature))*100
        
        num_dataset_li.append(num_dataset)
        num_feature_li.append(num_feature)
        usage_li.append(usage)

        # ---------------------
        #  Construct new train data with Quilt result 
        # ---------------------
        train_ds, valid_ds, test_ds = prepare_data(n, x_all_cali, y_all_cali, concept_drifts, 
                                                   dataset_final, feature_final)
        train_loader = DataLoader(train_ds, batch_size=128, shuffle=True)
        valid_loader = DataLoader(valid_ds, batch_size=128, shuffle=True)
        test_loader = DataLoader(test_ds, batch_size=128, shuffle=True)

        # ---------------------
        #  Initialize model, optimizer, and criterion
        # ---------------------
        model = NormalNN(input_features=len(feature_final), seed=s)
        model = model.cuda()
        optimizer_config = {"lr": lr}
        clf = NNClassifier(model, nn.BCELoss(reduction='mean'), optim.Adam, optimizer_config)
        
        # ---------------------
        #  Model training
        # ---------------------
        clf.fit({"train": train_loader, "val": valid_loader}, epochs=2000, earlystop_path='checkpoint_SEA_DSFSC.pt')
        
        test_output, test_loss = clf.evaluate(test_loader)
        test_score = accuracy_score(test_output['true_y'], test_output['output'])
        test_score_li.append(test_score)
        
    print('test acc: avg %f, std %f' %(np.mean(test_score_li), np.std(test_score_li)))
    all_test_acc.append(np.mean(test_score_li))
    all_test_std.append(np.std(test_score_li))
    
    print('data usage(#feature, #segment, %%total): (%f, %f, %f%%)' 
          %(np.mean(num_feature_li), np.mean(num_dataset_li), np.mean(usage_li)))
    all_dataset.append(np.mean(num_dataset_li))
    all_feature.append(np.mean(num_feature_li))
    all_usage.append(np.mean(usage_li))
    
    print("----------------------------------------------------------------------------")
    
print('overall test acc: avg %f, std %f' %(np.mean(all_test_acc), np.mean(all_test_std)))
print('overall data usage(#feature, #segment, %%total): (%f, %f, %f%%)' 
      %(np.mean(all_feature), np.mean(all_dataset), np.mean(all_usage)))

test segment:  7
test acc: avg 0.855222, std 0.014549
data usage(#feature, #segment, %total): (2.400000, 2.600000, 30.476190%)
----------------------------------------------------------------------------
test segment:  8
test acc: avg 0.806444, std 0.016565
data usage(#feature, #segment, %total): (2.600000, 3.000000, 32.500000%)
----------------------------------------------------------------------------
test segment:  9
test acc: avg 0.898333, std 0.003833
data usage(#feature, #segment, %total): (2.000000, 3.400000, 25.185185%)
----------------------------------------------------------------------------
overall test acc: avg 0.853333, std 0.011649
overall data usage(#feature, #segment, %total): (2.333333, 3.000000, 29.387125%)
